# Constructing the full pipeline

We want to construct a full pipeline which is able to run batches of games at a time so that we can efficiently gather data. Let's start by running a benchmark on 10 games to test how fast are pipeline would currently be.

### Setup

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent 
sys.path.insert(0, str(project_root))
from src.simulation.board_runner import play_board

import pandas as pd
import time

def run_simulation(num_games: int = 10):
    game_records = pd.DataFrame(columns=[
        "game_id",
        "num_cols",
        "num_rows",
        "mine_count",
        "success_prob",
        "min_guess_prob",
        "max_guess_prob",
        "avg_guess_prob",
        "win_no_guess",
        "win",
        "first_click",
        "time_taken"
    ])

    guess_records = pd.DataFrame(columns=[
        "game_id",
        "guess_number",
        "guess_prob",
        "guess_result",
        "component_size",
        "num_valid_configs"
    ])

    sim_start_time = time.time()

    for i in range(num_games):
        run_start_time = time.time()
        # Play the board and get the game record
        game_record, guess_record = play_board(i)

        print(f"Game {i} played")

        run_end_time = time.time()

        # Add the time taken for this run to the game record
        game_record.time_taken = run_end_time - run_start_time

        # Append the game record to the DataFrame
        game_records = pd.concat([game_records, pd.DataFrame([game_record.__dict__])], ignore_index=True)
        # Append the guess records to the DataFrame
        guess_records = pd.concat([guess_records, pd.DataFrame([g.__dict__ for g in guess_record])], ignore_index=True)

    sim_end_time = time.time()

    print(f"Total time taken: {sim_end_time - sim_start_time:.2f} seconds")
    print(f"Average time per game: {(sim_end_time - sim_start_time) / 10:.2f} seconds")
    print(f"Longest time taken for a single game: {game_records['time_taken'].max():.2f} seconds")
    print(f"Shortest time taken for a single game: {game_records['time_taken'].min():.2f} seconds")
    print(game_records.head())
    print(game_records.describe())
    print(guess_records.head())
    print(guess_records.describe())

### Running the first simulation

In [2]:
run_simulation(num_games=10)

Game 0 played
Game 1 played
Game 2 played
Game 3 played
Game 4 played
Game 5 played
Game 6 played
Game 7 played
Game 8 played
Game 9 played
Total time taken: 9.08 seconds
Average time per game: 0.91 seconds
Longest time taken for a single game: 3.24 seconds
Shortest time taken for a single game: 0.11 seconds
  game_id num_cols num_rows mine_count success_prob min_guess_prob  \
0       0       30       30        270     0.308989            0.0   
1       1       30       30        270     0.861011       0.027289   
2       2       30       30        270     0.139005            0.0   
3       3       30       30        270     0.722096            0.0   
4       4       30       30        270     0.329396            0.0   

  max_guess_prob avg_guess_prob win_no_guess    win first_click time_taken  \
0        0.25025       0.173409        False  False     (20, 1)   0.473996   
1        0.08879        0.04822        False  False    (12, 24)   0.177043   
2       0.191802       0.086915    

As seen in the output, the disparity in time taken to run a board is large, ranging from 0.11 seconds to 3.24 seconds. At this rate, a full simulation of 10000 games will take around 2.7 hours, which we ideally want to be lower. We can explain the disparity by the variying component size and the time taken to enumerate a large component. Let's confirm this is the case.

In [4]:
import cProfile
cProfile.run("for s in range(10): play_board(s)", sort="cumulative")

         7946233 function calls (6053619 primitive calls) in 13.532 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000   13.531   13.531 {built-in method builtins.exec}
        1    0.003    0.003   13.531   13.531 <string>:1(<module>)
       10    0.006    0.001   13.528    1.353 board_runner.py:7(play_board)
     1112    0.102    0.000    9.266    0.008 probability_engine.py:265(_enumerate_component)
1893708/1112    9.021    0.000    9.090    0.008 probability_engine.py:320(search)
      188    0.041    0.000    7.233    0.038 probability_engine.py:16(probabilities)
      188    0.207    0.001    7.192    0.038 probability_engine.py:66(_mine_weights)
       94    0.001    0.000    4.755    0.051 probability_engine.py:53(best_guess)
       94    0.001    0.000    3.652    0.039 probability_engine.py:373(num_valid_configs)
       94    0.044    0.000    2.545    0.027 probability_engine.py:23(reso

As we can clearly see, almost all the time is spent running the search function which runs in the enumeration function. We need to find a way to reduce enumeration time, or maybe even skip enumeration as a whole if necessary. On top of this, we see how important optimising this is by the result from running the same boards with different start clicks. If we happen to choose a first click which does not lead to much deduction, we will end up with a heavier board computationally.

## Dynamic Cell Ordering

The old implementation made use of a fixed branching order which simply went through each cells in order and filled them in to compute whether the current configuration was valid. Our new implementation will decide which cell to fill in next using the minimum remaining undecided cells left. We compute the amount of undecided cells for each constraint at each iteration so we have a dynamic count for each, which allows us to decide the next cell. This should save time because we are able to skip invalid configurations faster by spotting them earlier, since we are not just simply going down each cell and computing it but actually looking to figure out whether a configuration is valid or not in the fastest way possible. 

With our new implementation, let's run the simulation again.

In [3]:
run_simulation(num_games=10)

Game 0 played
Game 1 played
Game 2 played
Game 3 played
Game 4 played
Game 5 played
Game 6 played
Game 7 played
Game 8 played
Game 9 played
Total time taken: 8.31 seconds
Average time per game: 0.83 seconds
Longest time taken for a single game: 3.43 seconds
Shortest time taken for a single game: 0.01 seconds
  game_id num_cols num_rows mine_count success_prob min_guess_prob  \
0       0       30       30        270     0.148062            0.0   
1       1       30       30        270     0.511828            0.0   
2       2       30       30        270     0.189439            0.0   
3       3       30       30        270     0.281978            0.0   
4       4       30       30        270     0.298828            0.0   

  max_guess_prob avg_guess_prob win_no_guess    win first_click time_taken  \
0       0.183529       0.118445        False  False     (1, 29)   0.622069   
1         0.2653       0.148948        False  False     (3, 26)   0.056389   
2       0.165088       0.067856    

From our new run, we can see that the new implementation doesn't make much difference in terms of execution speed, as the longest board still takes more than 3 seconds to run. Let's see if this comes down to our dynamic ordering taking more time, or simply because the enumeration is still too long. 

In [5]:
import cProfile
cProfile.run("for s in range(10): play_board(s)", sort="cumulative")

         30104030 function calls (26169698 primitive calls) in 27.086 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      2/1    0.000    0.000   26.975   26.975 {built-in method builtins.exec}
        1    0.004    0.004   26.975   26.975 <string>:1(<module>)
       10    0.118    0.012   25.529    2.553 board_runner.py:7(play_board)
     1263    0.026    0.000   22.236    0.018 probability_engine.py:266(_enumerate_component)
3935594/1263   10.865    0.000   22.202    0.018 probability_engine.py:325(search)
      258    0.066    0.000   17.700    0.069 probability_engine.py:16(probabilities)
      258    0.270    0.001   17.575    0.068 probability_engine.py:66(_mine_weights)
      129    0.002    0.000    9.136    0.071 probability_engine.py:53(best_guess)
      129    0.064    0.000    8.671    0.067 probability_engine.py:23(resolve)
  1252959    3.524    0.000    6.697    0.000 {built-in method builtins.sum}
      1

We see that a lot of time is lost on picking the next cell dynamically, which means that maybe it is not the best way to go.

### Running boards in parallel

A guaranteed way to speed up processing is running multiple boards concurrently. This means that even when one core is spending a long time to solve one board, the other cores can solve multiple easy boards at the same time.

First, we will try our current new implementation with parallel processing on a 100 game batch.

In [ ]:
import multiprocessing as mp

def run_parallel(num_games: int, num_workers: int | None = None, per_game_timeout: float = 15.0) -> None:
    """Time running `num_games` games in parallel, printing as each finishes.
    Forcibly kills any worker still running past `per_game_timeout` seconds,
    so one stuck board can't hang the whole benchmark.
    """
    pool = mp.Pool(processes=num_workers)
    start = time.perf_counter()
    async_results = {seed: pool.apply_async(play_board, (seed,)) for seed in range(num_games)}

    completed, stuck = 0, []
    try:
        for seed, result in async_results.items():
            try:
                result.get(timeout=per_game_timeout)
                completed += 1
                print(f"board {seed} done ({time.perf_counter() - start:.2f}s elapsed)", flush=True)
            except mp.TimeoutError:
                stuck.append(seed)
                print(f"board {seed} exceeded {per_game_timeout}s — likely stuck", flush=True)
    except KeyboardInterrupt:
        print("Interrupted — killing all workers now...", flush=True)
    finally:
        pool.terminate()  # forcibly kills all worker processes, stuck or not
        pool.join()

    elapsed = time.perf_counter() - start
    print(f"\n{completed}/{num_games} completed in {elapsed:.2f}s"
          + (f" ({len(stuck)} stuck: {stuck})" if stuck else ""))



In [8]:
run_parallel(num_games=10)

board 0 done (0.40s elapsed)
board 1 done (0.40s elapsed)
board 2 done (0.53s elapsed)
board 3 done (0.53s elapsed)
board 4 done (1.89s elapsed)
board 5 done (1.89s elapsed)
board 6 done (1.89s elapsed)
board 7 done (1.89s elapsed)
board 8 done (8.30s elapsed)
board 9 done (8.31s elapsed)

10/10 completed in 8.35s


Running boards in parallel seems to make easier boards much faster to run, however still clogs the simulation on the larger, heavier boards. To combat this, I think the best solution is to run a pipeline which will try seeds until we have 50000 games worth of data. If any board exceeds 10 seconds of computation, we can try a different random first click to see if this reveals components differently and leads to a nicer board to compute (Note that taking another random first click can techincally lead to the same click, but is very unlikely so just picking randomly again is sufficient). If this doesn't work 3 times, we can skip the board and move on. This will result in capturing data for most boards, but not getting hung up on difficult boards. Hopefully this should speed the pipeline up.

### Full batch runner

Let's test our new batch runner on a batch of 100 to see how it performs and if the outputs are correct.

In [3]:
from src.simulation.full_sim_pipeline import run_batch

### Test run

In [4]:
run_batch(num_games=100, num_workers=4, chunk_size=50, out_dir="../data/raw/test")

Starting 100 games, 4 workers, 10.0s/attempt, max 3 attempts/board
[    17.2s] completed 50/100 (skipped 0)
board 57: timeout (attempt 1), retrying with a new first click
board 73: timeout (attempt 1), retrying with a new first click
board 99: timeout (attempt 1), retrying with a new first click
board 99: timeout (attempt 2), retrying with a new first click
[    55.5s] completed 100/100 (skipped 0)

Done: 100 completed, 0 skipped, 0.9 min total


Our test run seems to be working fine. Let's check that the data we have seems alright.

In [11]:
games = pd.read_parquet("../data/raw/test/games")       # reads all games_*.parquet as one df
decisions = pd.read_parquet("../data/raw/test/decisions")

print(games.shape, decisions.shape)
print(games.head())
print(decisions.head())
print(games.info())
print(games.isna().sum())
print(games["game_id"].is_unique)

(100, 12) (876, 6)
  game_id  num_cols  num_rows  mine_count  num_guesses  success_prob  \
0       1        30        30         270            1      0.699317   
1       0        30        30         270            4      0.500309   
2       2        30        30         270            6      0.601438   
3       4        30        30         270            7      0.548429   
4       6        30        30         270            4      0.552454   

   min_guess_prob  max_guess_prob  avg_guess_prob  win_no_guess    win  \
0        0.300683        0.300683        0.300683         False  False   
1        0.101719        0.179710        0.158355         False  False   
2        0.000000        0.120387        0.080231         False  False   
3        0.000000        0.134306        0.081406         False  False   
4        0.000000        0.297313        0.127778         False  False   

  first_click  
0     [18, 0]  
1     [23, 1]  
2     [4, 14]  
3     [4, 17]  
4    [25, 20]  
  game_

The data follows the correct shape and seems to store the right data. Now lets check that the data values for our features make sense.

In [9]:
assert (games["success_prob"] >= 0).all() and (games["success_prob"] <= 1).all()
assert (games["num_guesses"] >= 0).all()
assert (decisions["guess_prob"] >= 0).all() and (decisions["guess_prob"] <= 1).all()
assert (games["mine_count"] > 0).all()

Now we can check that the decisions and games are consistent with each other.

In [ ]:
guess_counts_from_decisions = decisions.groupby("game_id").size()
guess_counts_from_games = games.set_index("game_id")["num_guesses"]

mismatch = guess_counts_from_games[guess_counts_from_games.index.isin(guess_counts_from_decisions.index)] \
    .compare(guess_counts_from_decisions.reindex(guess_counts_from_games.index).fillna(0))
print(mismatch)  # should be empty

Empty DataFrame
Columns: [self, other]
Index: []


This confirms that the data is correct. We are now ready to run the full 50000 board pipeline.

### Full Pipeline

In [4]:
run_batch(num_games=50000, num_workers=6, print_every=1000)

Starting 50000 games, 6 workers, 10.0s/attempt, max 3 attempts/board
[   351.3s] completed 1000/50000 (skipped 0)
board 1874: skipped after 3 attempts (timeout)
[   699.7s] completed 2000/50000 (skipped 1)
board 2066: skipped after 3 attempts (timeout)
[  1034.2s] completed 3000/50000 (skipped 2)
board 3335: skipped after 3 attempts (timeout)
board 3650: skipped after 3 attempts (timeout)
board 3772: skipped after 3 attempts (timeout)
[  1411.2s] completed 4000/50000 (skipped 5)
board 4676: skipped after 3 attempts (timeout)
board 4885: skipped after 3 attempts (timeout)
[  1788.3s] completed 5000/50000 (skipped 7)
[  2152.1s] completed 6000/50000 (skipped 7)
board 6223: skipped after 3 attempts (timeout)
board 6451: skipped after 3 attempts (timeout)
[  2537.6s] completed 7000/50000 (skipped 9)
board 7890: skipped after 3 attempts (timeout)
[  2912.5s] completed 8000/50000 (skipped 10)
[  3275.7s] completed 9000/50000 (skipped 10)
[  3664.2s] completed 10000/50000 (skipped 10)
board 1

With our new model, only 51 boards are skipped out of 50000, leaving us with 499949 boards worth of raw data. The pipeline took around 6 hours, which compared to our beginning pipeline which would have taken around 13.5 hours, is more than a 50% reduction. On top of this, we only used 6 workers for this pipeline. A more powerful machine which would be able to run on 8 cores in parallel would complete even faster. 

Let's quickly confirm that the data we have is correct, before moving on to an EDA and feature engineering.

In [ ]:
full_games = pd.read_parquet("../data/raw/games")       # reads all games_*.parquet as one df
full_decisions = pd.read_parquet("../data/raw/decisions")


print(full_games.shape, full_decisions.shape)
print(full_games.head())
print(full_decisions.head())
print(full_games.info()) 
print(full_games.isna().sum())
print(full_games["game_id"].is_unique)

(49949, 14) (444589, 10)
  game_id  num_cols  num_rows  mine_count  remaining_mines  num_guesses  \
0       3        30        30         270              264            2   
1       1        30        30         270              260            2   
2       5        30        30         270              215            7   
3       2        30        30         270              105           14   
4       6        30        30         270              194            4   

   success_prob  min_guess_prob  max_guess_prob  avg_guess_prob  win_no_guess  \
0      0.810429        0.082987        0.116230        0.099608         False   
1      0.666564        0.057487        0.292780        0.175134         False   
2      0.532640        0.000000        0.157593        0.084183         False   
3      0.377471        0.000000        0.159591        0.065192         False   
4      0.768704        0.000000        0.142372        0.061515         False   

     win first_click  cells_revealed_

This confirms that our data is ready to use.

### Summary

We have constructed a pipeline which runs boards in batches and stores data in separate parquet files, which are ready to be processed. After running the full pipeline on 50000 boards, with a 99.9% completion rate, we have more than enough data to perform some in depth analysis and move on to the next stage of the project, which will finally aim to provide an answer the initial question of the project.